In [0]:
import pandas as pd
from pyspark.sql import functions as F

In [0]:
df = spark.read.csv("s3a://data5035-spring26/drone_data.csv", header=True, inferSchema=True)
display(df)

In [0]:
# Consolidating all imports and reusable functions at the top of the notebook
from pyspark.sql.functions import col, count, when, countDistinct, lit

def check_duplicates(df):
    """Returns duplicate rows and their occurrence count."""
    return df.groupBy(df.columns).count().filter("count > 1")

def check_nulls(df):
    """Returns the count of null values for every column in the DataFrame."""
    return df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])

def check_unique_counts(df):
    """Returns the number of distinct values for every column."""
    return df.select([countDistinct(c).alias(c) for c in df.columns])

def inspect_values(df, column_name):
    """Groups by a column, counts occurrences, and sorts alphabetically to reveal spelling variations."""
    return df.groupBy(column_name).count().orderBy(col(column_name).asc())

In [0]:
dupecheck = display(check_duplicates(df))
nullcheck = display(check_nulls(df))

print(f'Duplicates: {dupecheck}')
print(f'Nulls: {nullcheck}')

In [0]:
uniquecheck = display(check_unique_counts(df))
print(f'Unique Counts: {uniquecheck}')

# Found two GPS Unit numbers


In [0]:
gps_units_unique = inspect_values(df, 'GPS_UNIT_NUMBER')
display(gps_units_unique)
print(f'GPS Units: {gps_units_unique.count()}')

gamma_units_unique = inspect_values(df, 'GAMMA_DETECTOR_UNIT_NUMBER')
display(gamma_units_unique)
print(f'Gamma Detector Units: {gamma_units_unique.count()}')

cesium_units_unique = inspect_values(df, 'CESIUM_137_DETECTOR_UNIT_NUMBER')
display(cesium_units_unique)
print(f'Cesium Detector Units: {cesium_units_unique.count()}')

thorium_units_unique = inspect_values(df, 'THORIUM_232_DETECTOR_UNIT_NUMBER')
display(thorium_units_unique)
print(f'Thorium Detector Units: {thorium_units_unique.count()}')

In [0]:
# Review GPS_LAT and GPS_LNG ranges and potential outliers
gps_stats = df.select(
    F.min('GPS_LAT').alias('min_lat'),
    F.max('GPS_LAT').alias('max_lat'),
    F.min('GPS_LNG').alias('min_lng'),
    F.max('GPS_LNG').alias('max_lng'),
    F.mean('GPS_LAT').alias('mean_lat'),
    F.mean('GPS_LNG').alias('mean_lng'),
    F.stddev('GPS_LAT').alias('std_lat'),
    F.stddev('GPS_LNG').alias('std_lng')
)
display(gps_stats)

# Identify potential outliers using 3 standard deviations from the mean
lat_stats = gps_stats.collect()[0]
lat_mean = lat_stats['mean_lat']
lat_std = lat_stats['std_lat']
lng_mean = lat_stats['mean_lng']
lng_std = lat_stats['std_lng']

outliers = df.filter(
    (col('GPS_LAT') < lat_mean - 3 * lat_std) | (col('GPS_LAT') > lat_mean + 3 * lat_std) |
    (col('GPS_LNG') < lng_mean - 3 * lng_std) | (col('GPS_LNG') > lng_mean + 3 * lng_std)
)
display(outliers.select('GPS_LAT', 'GPS_LNG'))

In [0]:
detector_stats = df.select(
    F.min('GAMMA_LEVEL').alias('min_gamma'),
    F.max('GAMMA_LEVEL').alias('max_gamma'),
    F.mean('GAMMA_LEVEL').alias('avg_gamma'),
    F.min('CESIUM_137_LEVEL').alias('min_ces'),
    F.max('CESIUM_137_LEVEL').alias('max_ces'),
    F.mean('CESIUM_137_LEVEL').alias('avg_ces'),
    F.mean('THORIUM_232_LEVEL').alias('mean_thor'),
    F.mean('THORIUM_232_LEVEL').alias('mean_thor'),
    F.mean('THORIUM_232_LEVEL').alias('avg_thor')
)

display(detector_stats)

Note that changes to detector calibration are slower than the rest of the transactional data. They change, but not often; and they aren't necessarily a metric of the sampling as much as they are a metric of the process.

In [0]:
%pip install folium

In [0]:
import pandas as pd
import folium

# Convert Spark DataFrame to Pandas DataFrame for mapping
gps_df = df.select('GPS_LAT', 'GPS_LNG').dropna().toPandas()

# Center map at mean lat/lng
center_lat = gps_df['GPS_LAT'].mean()
center_lng = gps_df['GPS_LNG'].mean()
m = folium.Map(location=[center_lat, center_lng], zoom_start=10)

# Add points to map
for _, row in gps_df.iterrows():
    folium.CircleMarker(
        location=[row['GPS_LAT'], row['GPS_LNG']],
        radius=2,
        color='blue',
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

display(m)

# Star Schema Notes

**FACT** represent business metric aggregates

**DIMENSIONS** represent entities used to collect measurements:

- **DIM1: TIME**
(reports when something happens)
  - RowID
  - DATETIME

- **DIM2: GPS_UNIT**
(reports latitudinal and logitudinal data)
  - GPSID
  - LAT
  - LONG

- **DIM3: DETECTOR_UNIT**
(reports radiation detections made)
  - DETECTOR_ID
  - NAME
  - TYPE
  - BRAND
  - UNIT
  - CALIBRATION_PRECISION

- **DIM4: CALIBRATION**
(slower changing data, related to machines)
  - MAINT_ID
  - DETECTOR_ID
  - GPS_UNIT
  - UNIT_CALIBRATED
  - TIME_CALIBRATED
  - LAST_FRAMEWARE_UPDATE
  - FRAMEWARE_UPDATE_NEEDED
  - 
  - 
  - 
  

  